In [0]:


# BLOCK 1 — VALIDATION CONFIGURATION
# ===================================================

from datetime import datetime, timezone

from pyspark.sql import functions as F


"""
Validate the Bronze production-lot table against the approved five-year
dataset control totals.

The baseline includes deliberately injected duplicate and unknown-device
records that must remain present in Bronze for downstream quality testing.
"""

TARGET_TABLE = "semiconplus_portfolio.bronze.production_lots"

EXPECTED_ROW_COUNT = 18_126
EXPECTED_DISTINCT_LOT_COUNT = 18_125
EXPECTED_DUPLICATE_RECORD_COUNT = 1
EXPECTED_SOURCE_FILE_COUNT = 60
EXPECTED_UNKNOWN_DEVICE_COUNT = 1
EXPECTED_MIN_DATE = "2021-01-01"
EXPECTED_MAX_DATE = "2025-12-31"

VALIDATION_TIME_UTC = datetime.now(timezone.utc)

print(f"Validating: {TARGET_TABLE}")
print(f"Validation time UTC: {VALIDATION_TIME_UTC.isoformat()}")

In [0]:
# ===================================================
# BLOCK 2 — TABLE AVAILABILITY
# ===================================================

"""
Confirm that the Bronze production-lot table is available before running
reconciliation and data-quality controls.
"""

table_exists = spark.catalog.tableExists(TARGET_TABLE)

print(f"Table exists: {table_exists}")

assert table_exists, f"Required Bronze table does not exist: {TARGET_TABLE}"

In [0]:
# ===================================================
# BLOCK 3 — SCHEMA AND SAMPLE REVIEW
# ===================================================

"""
Load the Bronze table and expose its schema and sample records for
execution review and validation evidence.
"""

bronze_df = spark.table(TARGET_TABLE)

bronze_df.printSchema()

display(
    bronze_df
    .orderBy("production_date", "lot_id")
    .limit(20)
)

In [0]:
# ===================================================
# BLOCK 4 — VALIDATION METRICS
# ===================================================

"""
Calculate completeness, uniqueness, file coverage, date coverage,
reference-quality, and rescued-data metrics for the Bronze dataset.
"""

metrics = (
    bronze_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("lot_id").alias("distinct_lot_count"),

        # Measure excess records against the production-lot business key.
        (
            F.count("*") - F.countDistinct("lot_id")
        ).alias("duplicate_record_count"),

        F.countDistinct("_source_file_path").alias("source_file_count"),

        # Measure the deliberate failed device-reference test record.
        F.sum(
            F.when(
                F.col("device_id") == "UNKNOWN_DEVICE",
                F.lit(1),
            ).otherwise(F.lit(0))
        ).alias("unknown_device_count"),

        F.min("production_date").alias("minimum_production_date"),
        F.max("production_date").alias("maximum_production_date"),

        # Identify records containing fields outside the approved schema.
        F.sum(
            F.when(
                F.col("_rescued_data").isNotNull(),
                F.lit(1),
            ).otherwise(F.lit(0))
        ).alias("rescued_record_count"),
    )
    .first()
)

validation_metrics = metrics.asDict()

for metric_name, metric_value in validation_metrics.items():
    print(f"{metric_name}: {metric_value}")


In [0]:
# ===================================================
# BLOCK 5 — CONTROL-TOTAL ASSERTIONS
# ===================================================

"""
Enforce the approved Bronze control totals.

Any failed assertion blocks downstream Silver processing until the
source data or ingestion pipeline has been investigated.
"""

expected_metrics = {
    "row_count": EXPECTED_ROW_COUNT,
    "distinct_lot_count": EXPECTED_DISTINCT_LOT_COUNT,
    "duplicate_record_count": EXPECTED_DUPLICATE_RECORD_COUNT,
    "source_file_count": EXPECTED_SOURCE_FILE_COUNT,
    "unknown_device_count": EXPECTED_UNKNOWN_DEVICE_COUNT,
    "minimum_production_date": EXPECTED_MIN_DATE,
    "maximum_production_date": EXPECTED_MAX_DATE,
    "rescued_record_count": 0,
}

for metric_name, expected_value in expected_metrics.items():
    actual_value = validation_metrics[metric_name]
    assert actual_value == expected_value, (
        f"Validation failed for {metric_name}: "
        f"expected {expected_value}, found {actual_value}."
    )

print("Bronze control-total validation passed.")

In [0]:
# ===================================================
# BLOCK 6 — METADATA VALIDATION
# ===================================================

"""
Verify that every Bronze record contains the metadata required for
file-level traceability and pipeline-run investigation.
"""

metadata_columns = [
    "_source_file_path",
    "_source_file_name",
    "_source_file_modification_time",
    "_ingested_at_utc",
    "_pipeline_run_id",
]

metadata_aggregations = [
    F.sum(
        F.when(F.col(column_name).isNull(), 1).otherwise(0)
    ).alias(f"null_{column_name.lstrip('_')}")
    for column_name in metadata_columns
]

metadata_metrics = (
    bronze_df
    .agg(*metadata_aggregations)
    .first()
    .asDict()
)

for metric_name, metric_value in metadata_metrics.items():
    print(f"{metric_name}: {metric_value}")
    assert metric_value == 0, (
        f"Metadata validation failed: {metric_name} = {metric_value}"
    )

print("Bronze metadata validation passed.")

In [0]:
# ===================================================
# BLOCK 7 — DUPLICATE INSPECTION
# ===================================================

"""
Identify production-lot business keys occurring more than once.

The expected duplicate remains in Bronze. Silver processing will apply
the approved deduplication rule and record the rejected occurrence.
"""

duplicate_lots = (
    bronze_df
    .groupBy("lot_id")
    .agg(F.count("*").alias("record_count"))
    .filter(F.col("record_count") > 1)
)

display(duplicate_lots)

assert duplicate_lots.count() == 1, (
    "Expected exactly one duplicated lot ID."
)

assert duplicate_lots.first()["record_count"] == 2, (
    "The duplicated lot ID must contain exactly two records."
)

In [0]:
# ===================================================
# BLOCK 8 — UNKNOWN-DEVICE INSPECTION
# ===================================================

"""
Identify production-lot records that intentionally fail the device
reference-data relationship.

The record remains unchanged in Bronze and will be routed to the Silver
quarantine process.
"""

unknown_device_records = bronze_df.filter(
    F.col("device_id") == "UNKNOWN_DEVICE"
)

display(unknown_device_records)

assert unknown_device_records.count() == 1, (
    "Expected exactly one deliberate unknown-device record."
)

In [0]:

# ===================================================
# BLOCK 9 — FILE-LEVEL RECONCILIATION
# ===================================================

"""
Reconcile record counts and date coverage by source file.

This control detects missing files, unexpected duplicate ingestion,
incorrect file content, and incomplete monthly coverage.
"""

records_by_file = (
    bronze_df
    .groupBy("_source_file_name", "_source_file_path")
    .agg(
        F.count("*").alias("record_count"),
        F.min("production_date").alias("minimum_date"),
        F.max("production_date").alias("maximum_date"),
    )
    .orderBy("_source_file_name")
)

display(records_by_file)

assert records_by_file.count() == EXPECTED_SOURCE_FILE_COUNT, (
    "Not all expected monthly source files are represented in Bronze."
)

assert records_by_file.filter(F.col("record_count") <= 0).count() == 0, (
    "One or more source files produced no Bronze records."
)

print("Source-file reconciliation passed.")

In [0]:
# ===================================================
# BLOCK 10 — VALIDATION SUMMARY
# ===================================================
"""
Publish the final Bronze validation result for pipeline execution logs
and project documentation.
"""

validation_result = {
    "table": TARGET_TABLE,
    "status": "PASSED",
    "validated_at_utc": VALIDATION_TIME_UTC.isoformat(),
    **validation_metrics,
}

print("BRONZE VALIDATION PASSED")

for key, value in validation_result.items():
    print(f"{key}: {value}")

In [0]:
# ===================================================
# BLOCK 11 — CAPTURE PRE-RERUN COUNT
# ===================================================

"""
Capture the validated Bronze count before repeating the ingestion
pipeline with the existing checkpoint.
"""

count_before_rerun = spark.table(TARGET_TABLE).count()

print(f"Count before rerun: {count_before_rerun:,}")

assert count_before_rerun == EXPECTED_ROW_COUNT

In [0]:

# ---------------------
# IDEMPOTENCY PROCEDURE
# ---------------------

# 1. Run Block 11 and leave count_before_rerun in the active notebook session.
# 2. Open 01_bronze_production_lots and select Run all.
# 3. Return to this validation notebook without restarting its Python session.
# 4. Run Block 12 below.

# ===================================================
# BLOCK 12 — IDEMPOTENCY VALIDATION
# ===================================================

"""
Verify that the existing checkpoint prevents previously processed source
files from being appended again.
"""

count_after_rerun = spark.table(TARGET_TABLE).count()
rows_added_by_rerun = count_after_rerun - count_before_rerun

print(f"Count before rerun: {count_before_rerun:,}")
print(f"Count after rerun: {count_after_rerun:,}")
print(f"Rows added by rerun: {rows_added_by_rerun:,}")

assert count_after_rerun == EXPECTED_ROW_COUNT
assert rows_added_by_rerun == 0

print("IDEMPOTENCY TEST PASSED")

In [0]:
%sql
DESCRIBE DETAIL semiconplus_portfolio.bronze.production_lots;

In [0]:
%sql
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT lot_id) AS distinct_lot_count,
    COUNT(*) - COUNT(DISTINCT lot_id) AS duplicate_record_count,
    COUNT(DISTINCT _source_file_path) AS source_file_count,
    COUNT_IF(device_id = 'UNKNOWN_DEVICE') AS unknown_device_count,
    COUNT_IF(_rescued_data IS NOT NULL) AS rescued_record_count,
    MIN(production_date) AS minimum_production_date,
    MAX(production_date) AS maximum_production_date
FROM semiconplus_portfolio.bronze.production_lots;
